# Strategic Allocation – Interactive Plotly Visualizations

Replaces gut-feel order cutting with a **rules-based tier allocation model**.  
All charts render **locally inside Jupyter — no login required**.

**Scenario:** Beefsteak Tomatoes — Demand = 8,000 cases, Supply = 6,000 cases (−2,000 shortage)  
**Rule:** Tier 1 = 0% cut · Tier 2 = 20% cut · Tier 3 = 50% cut

**Run order:** Cell 1 → Cell 2 → Cell 3

In [ ]:
# ── Cell 1 · Imports ──────────────────────────────────────────────────────────
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Cell 2 · Build Strategic Allocation DataFrame ─────────────────────────────

# ── 2.1  Scenario constants ───────────────────────────────────────────────────
SKU_NAME      = "Beefsteak Tomatoes (15lb 22ct No1)"
TOTAL_DEMAND  = 8_000
TOTAL_SUPPLY  = 6_000
SHORTAGE      = TOTAL_DEMAND - TOTAL_SUPPLY    # 2,000 cases
FLAT_CUT_RATE = SHORTAGE / TOTAL_DEMAND        # 25% uniform cut baseline

CUT_RATE   = {"Tier 1 - High Margin":     0.00,
              "Tier 2 - Standard":         0.20,
              "Tier 3 - Flexible/Promo":   0.50}

UNIT_PRICE = {"Tier 1 - High Margin":    24.00,
              "Tier 2 - Standard":        20.00,
              "Tier 3 - Flexible/Promo":  16.00}

# ── 2.2  Customer order data ──────────────────────────────────────────────────
# Tier 1 total = 2,500  →  cut 0%   = 0 cases
# Tier 2 total = 2,500  →  cut 20%  = 500 cases
# Tier 3 total = 3,000  →  cut 50%  = 1,500 cases
# Total cuts   = 2,000  ✓
raw = [
    ("Metro Ontario Inc",           "Tier 1 - High Margin",      900),
    ("Kroger National",             "Tier 1 - High Margin",     1000),
    ("Walmart Premium Account",     "Tier 1 - High Margin",      600),
    ("Meijer Inc",                  "Tier 2 - Standard",         700),
    ("Sobeys Inc",                  "Tier 2 - Standard",         600),
    ("Loblaws Regional",            "Tier 2 - Standard",         650),
    ("Safeway West",                "Tier 2 - Standard",         550),
    ("Discount Foods East",         "Tier 3 - Flexible/Promo",   600),
    ("Food Service Distributor A",  "Tier 3 - Flexible/Promo",   700),
    ("Promo / Secondary Buyer",     "Tier 3 - Flexible/Promo",   500),
    ("Clearance Account",           "Tier 3 - Flexible/Promo",   600),
    ("Donation / Spot Market",      "Tier 3 - Flexible/Promo",   600),
]

df = pd.DataFrame(raw, columns=["Customer_Name", "Priority_Tier", "Original_Order_Qty"])

# ── 2.3  Allocation logic ─────────────────────────────────────────────────────
df["Suggested_Cut_Qty"]    = (df["Original_Order_Qty"] * df["Priority_Tier"].map(CUT_RATE)).round().astype(int)
df["Final_Allocated_Qty"]  = df["Original_Order_Qty"] - df["Suggested_Cut_Qty"]
df["Unit_Price"]            = df["Priority_Tier"].map(UNIT_PRICE)
df["Flat_Allocated_Qty"]   = (df["Original_Order_Qty"] * (1 - FLAT_CUT_RATE)).round().astype(int)
df["Revenue_Saved"]         = ((df["Final_Allocated_Qty"] - df["Flat_Allocated_Qty"]) * df["Unit_Price"]).round(2)
df["Fill_Rate_Pct"]         = (df["Final_Allocated_Qty"] / df["Original_Order_Qty"] * 100).round(1)

assert df["Suggested_Cut_Qty"].sum()   == SHORTAGE,     "Cut total mismatch"
assert df["Final_Allocated_Qty"].sum() == TOTAL_SUPPLY, "Allocation total mismatch"

print(f"Scenario validated  |  Demand: {TOTAL_DEMAND:,}  Supply: {TOTAL_SUPPLY:,}  Shortage: {SHORTAGE:,}")
print(f"Revenue saved vs flat-cut baseline: ${df[df['Revenue_Saved']>0]['Revenue_Saved'].sum():,.0f}")
df

In [ ]:
# ── Cell 3 · Four Interactive Plotly Charts ───────────────────────────────────
# No login. All charts render inline in Jupyter.
# Hover for tooltips · scroll to zoom · click legend to isolate series

TEMPLATE = "plotly_white"
RED      = "#D7263D"
GREEN    = "#2DC653"
GREY     = "#BDBDBD"
BLUE     = "#1F77B4"

TIER_COLORS = {
    "Tier 1 - High Margin":    "#1A6B3C",
    "Tier 2 - Standard":       "#F4A261",
    "Tier 3 - Flexible/Promo": "#D7263D",
}


# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 – Supply vs Demand by Tier
# Answers: How big is the gap and how is it absorbed across tiers?
# ─────────────────────────────────────────────────────────────────────────────
tier_summary = (
    df.groupby("Priority_Tier", sort=False)
    .agg(Ordered   =("Original_Order_Qty",  "sum"),
         Allocated =("Final_Allocated_Qty", "sum"),
         Cut       =("Suggested_Cut_Qty",   "sum"))
    .reset_index()
)

fig1 = go.Figure()
for _, row in tier_summary.iterrows():
    fig1.add_trace(go.Bar(
        name=f"{row['Priority_Tier']} – Allocated",
        x=[row["Priority_Tier"]],
        y=[row["Allocated"]],
        marker_color=TIER_COLORS[row["Priority_Tier"]],
        text=[f"{row['Allocated']:,}"],
        textposition="inside",
        hovertemplate=(
            f"<b>{row['Priority_Tier']}</b><br>"
            f"Ordered: {row['Ordered']:,}<br>"
            f"Allocated: {row['Allocated']:,}<br>"
            f"Cut: {row['Cut']:,} cases<extra></extra>"
        ),
    ))
    if row["Cut"] > 0:
        fig1.add_trace(go.Bar(
            name=f"{row['Priority_Tier']} – Cut",
            x=[row["Priority_Tier"]],
            y=[row["Cut"]],
            marker_color=GREY,
            marker_pattern_shape="/",
            text=[f"-{row['Cut']:,} cut"],
            textposition="inside",
            hovertemplate=f"Cut: {row['Cut']:,} cases<extra></extra>",
        ))

fig1.update_layout(
    title=dict(
        text=(
            f"<b>Chart 1 – Supply vs Demand by Tier</b><br>"
            f"<sup>Total Demand: {TOTAL_DEMAND:,} cases  |  "
            f"Total Supply: {TOTAL_SUPPLY:,} cases  |  "
            f"Shortage: {SHORTAGE:,} cases ({SHORTAGE/TOTAL_DEMAND:.0%})</sup>"
        ),
        x=0,
    ),
    barmode="stack",
    xaxis_title="Priority Tier",
    yaxis_title="Cases",
    template=TEMPLATE,
    showlegend=False,
    height=420,
)
fig1.show()


# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 – Original Order vs Final Allocation per Customer
# Answers: Exactly who gets cut, and by how much?
# ─────────────────────────────────────────────────────────────────────────────
df_sorted = df.sort_values(["Priority_Tier", "Original_Order_Qty"], ascending=[True, False])

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    name="Original Order",
    x=df_sorted["Customer_Name"],
    y=df_sorted["Original_Order_Qty"],
    marker_color=GREY,
    opacity=0.5,
    hovertemplate="<b>%{x}</b><br>Original Order: %{y:,} cases<extra></extra>",
))
fig2.add_trace(go.Bar(
    name="Final Allocation",
    x=df_sorted["Customer_Name"],
    y=df_sorted["Final_Allocated_Qty"],
    marker_color=[TIER_COLORS[t] for t in df_sorted["Priority_Tier"]],
    text=df_sorted["Fill_Rate_Pct"].astype(str) + "%",
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Allocated: %{y:,} cases<extra></extra>",
))
fig2.update_layout(
    title="<b>Chart 2 – Original Order vs Final Allocation per Customer</b>",
    barmode="overlay",
    yaxis_title="Cases",
    template=TEMPLATE,
    hovermode="x unified",
    height=460,
    xaxis_tickangle=-35,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig2.show()


# ─────────────────────────────────────────────────────────────────────────────
# CHART 3 – Revenue Saved vs Flat-Cut Baseline
# Answers: What is the financial case for tier-based allocation?
# ─────────────────────────────────────────────────────────────────────────────
df_rev = df.sort_values("Revenue_Saved", ascending=True).copy()
df_rev["Bar_Color"] = df_rev["Revenue_Saved"].apply(lambda v: GREEN if v > 0 else RED)
df_rev["Label"]     = df_rev["Revenue_Saved"].apply(
    lambda v: f"+${v:,.0f}" if v >= 0 else f"-${abs(v):,.0f}"
)
net_saved = df[df["Revenue_Saved"] > 0]["Revenue_Saved"].sum()

fig3 = go.Figure(go.Bar(
    x=df_rev["Revenue_Saved"],
    y=df_rev["Customer_Name"],
    orientation="h",
    marker_color=df_rev["Bar_Color"],
    text=df_rev["Label"],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Revenue vs flat-cut: $%{x:,.0f}<extra></extra>",
))
fig3.add_vline(x=0, line_color="black", line_width=1.5,
               annotation_text="Flat-cut baseline", annotation_position="top")
fig3.update_layout(
    title=dict(
        text=(
            f"<b>Chart 3 – Revenue Saved vs Flat-Cut Baseline per Customer</b><br>"
            f"<sup>Net revenue protected for Tier 1: ${net_saved:,.0f}</sup>"
        ),
        x=0,
    ),
    xaxis_title="Revenue vs Flat-Cut Baseline ($)",
    template=TEMPLATE,
    height=460,
    yaxis={"autorange": "reversed"},
    margin=dict(r=120),
)
fig3.show()


# ─────────────────────────────────────────────────────────────────────────────
# CHART 4 – Shortage Absorption (donut) + Fill Rate per Customer (bar)
# Answers: Is the cut strategy fair and defensible by tier?
# NOTE: specs=[[ {"type":"domain"}, {"type":"xy"} ]] is required because
#       go.Pie needs a "domain" subplot, not the default "xy" subplot.
# ─────────────────────────────────────────────────────────────────────────────
fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Cut Cases by Tier", "Fill Rate by Customer"),
    column_widths=[0.38, 0.62],
    horizontal_spacing=0.12,
    specs=[[{"type": "domain"}, {"type": "xy"}]],
)

tier_cuts = (
    df.groupby("Priority_Tier", sort=False)["Suggested_Cut_Qty"]
    .sum().reset_index()
)
fig4.add_trace(
    go.Pie(
        labels=tier_cuts["Priority_Tier"],
        values=tier_cuts["Suggested_Cut_Qty"],
        hole=0.55,
        marker_colors=[TIER_COLORS[t] for t in tier_cuts["Priority_Tier"]],
        textinfo="label+percent",
        hovertemplate="<b>%{label}</b><br>Cases cut: %{value:,}<br>Share: %{percent}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=1,
)
fig4.add_annotation(
    text=f"<b>{SHORTAGE:,}</b><br>cases cut",
    x=0.18, y=0.5, xref="paper", yref="paper",
    showarrow=False, font=dict(size=13), align="center",
)

df_fr = df.sort_values(["Priority_Tier", "Fill_Rate_Pct"], ascending=[True, True])
fig4.add_trace(
    go.Bar(
        x=df_fr["Fill_Rate_Pct"],
        y=df_fr["Customer_Name"],
        orientation="h",
        marker_color=[TIER_COLORS[t] for t in df_fr["Priority_Tier"]],
        text=df_fr["Fill_Rate_Pct"].astype(str) + "%",
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Fill Rate: %{x}%<extra></extra>",
    ),
    row=1, col=2,
)
fig4.add_vline(x=100, line_dash="dash", line_color="black", row=1, col=2)
fig4.add_vline(x=75,  line_dash="dot",  line_color=GREY,
               annotation_text="Flat 75%", annotation_font_size=9,
               annotation_position="top", row=1, col=2)
fig4.update_layout(
    title="<b>Chart 4 – Shortage Absorption & Fill Rates by Customer</b>",
    template=TEMPLATE, height=480, showlegend=False,
)
fig4.update_xaxes(title_text="Fill Rate (%)", range=[0, 115], row=1, col=2)
fig4.update_yaxes(autorange="reversed", row=1, col=2)
fig4.show()